# Module 02: Pandas for Machine Learning
## Notebook 06: Feature Engineering and Dataset Preparation

Machine learning models require purely numerical input matrices with no missing values or raw text. Feature engineering is the process of transforming raw tabular attributes into mathematical representations that maximize an algorithm's predictive performance.

---

### Learning Objectives
By the end of this notebook, you will be able to:
1. Encode categorical variables using **One-Hot Encoding** (`pd.get_dummies`) and understand the **Dummy Variable Trap**.
2. Apply ordinal mapping for ranked categorical variables.
3. Discretize continuous features into bins using `pd.cut()` and `pd.qcut()`.
4. Detect and filter statistical outliers using the **Interquartile Range (IQR)** method.
5. Partition a cleaned DataFrame into the feature matrix $X$ and target vector $y$ ready for Scikit-Learn.

In [1]:
import pandas as pd
import numpy as np

print(f"Pandas version: {pd.__version__}")

Pandas version: 3.0.6


### 1. Categorical Encoding: One-Hot Encoding vs. Ordinal Mapping

#### A. One-Hot Encoding (`pd.get_dummies`)
Used for **nominal** categories with no inherent order (e.g. `'Red'`, `'Green'`, `'Blue'`).

> **The Dummy Variable Trap:**
> If a category has $K$ distinct values, creating $K$ dummy columns causes perfect multicollinearity ($x_1 + x_2 + ... + x_K = 1$), which destabilizes linear and logistic regression!
> **Always set `drop_first=True`** to create $K - 1$ binary indicator columns.

In [2]:
df_raw = pd.DataFrame({
    'City': ['Paris', 'London', 'Berlin', 'Paris', 'Berlin'],
    'Vehicle_Type': ['Sedan', 'SUV', 'Sedan', 'Truck', 'SUV'],
    'Satisfaction': ['Low', 'High', 'Medium', 'Medium', 'High'],
    'Income': [45000, 85000, 62000, 38000, 110000],
    'Purchased': [0, 1, 1, 0, 1]
})

print("Raw Dataset:\n", df_raw)

# One-Hot Encoding with drop_first=True
df_encoded = pd.get_dummies(df_raw, columns=['City', 'Vehicle_Type'], drop_first=True, dtype=int)
print("\nOne-Hot Encoded (avoiding dummy trap):\n", df_encoded)

Raw Dataset:
      City Vehicle_Type Satisfaction  Income  Purchased
0   Paris        Sedan          Low   45000          0
1  London          SUV         High   85000          1
2  Berlin        Sedan       Medium   62000          1
3   Paris        Truck       Medium   38000          0
4  Berlin          SUV         High  110000          1

One-Hot Encoded (avoiding dummy trap):
   Satisfaction  Income  Purchased  City_London  City_Paris  \
0          Low   45000          0            0           1   
1         High   85000          1            1           0   
2       Medium   62000          1            0           0   
3       Medium   38000          0            0           1   
4         High  110000          1            0           0   

   Vehicle_Type_Sedan  Vehicle_Type_Truck  
0                   1                   0  
1                   0                   0  
2                   1                   0  
3                   0                   1  
4                   0 

#### B. Ordinal / Label Mapping
Used for **ordinal** categories where an inherent hierarchy exists (e.g. `'Low' < 'Medium' < 'High'`).

In [3]:
satisfaction_order = {
    'Low': 1,
    'Medium': 2,
    'High': 3
}

df_encoded['Satisfaction_Rank'] = df_raw['Satisfaction'].map(satisfaction_order)
# Drop the original non-numeric column
df_encoded = df_encoded.drop(columns=['Satisfaction'])

print("Encoded DataFrame with Ordinal Ranking:\n", df_encoded)

Encoded DataFrame with Ordinal Ranking:
    Income  Purchased  City_London  City_Paris  Vehicle_Type_Sedan  \
0   45000          0            0           1                   1   
1   85000          1            1           0                   0   
2   62000          1            0           0                   1   
3   38000          0            0           1                   0   
4  110000          1            0           0                   0   

   Vehicle_Type_Truck  Satisfaction_Rank  
0                   0                  1  
1                   0                  3  
2                   0                  2  
3                   1                  2  
4                   0                  3  


---
### 2. Numerical Discretization (Binning): `cut` vs. `qcut`

- `pd.cut()`: Divides data into **equal-width** intervals (uniform bin spans).
- `pd.qcut()`: Divides data into **equal-frequency** quantiles (equal number of samples per bin).

In [4]:
# Equal-Width Binning: 3 income brackets
df_encoded['Income_Bracket'] = pd.cut(df_encoded['Income'], bins=3, labels=['Low_Income', 'Mid_Income', 'High_Income'])

# Equal-Frequency Binning: Quartiles
df_encoded['Income_Quartile'] = pd.qcut(df_encoded['Income'], q=3, labels=['Q1', 'Q2', 'Q3'])

print("Income Binning Comparison:\n", df_encoded[['Income', 'Income_Bracket', 'Income_Quartile']])

Income Binning Comparison:
    Income Income_Bracket Income_Quartile
0   45000     Low_Income              Q1
1   85000     Mid_Income              Q3
2   62000     Low_Income              Q2
3   38000     Low_Income              Q1
4  110000    High_Income              Q3


---
### 3. Outlier Detection via Interquartile Range (IQR)

In non-Gaussian distributions, extreme outliers distort linear regression slopes and variance calculations.
The **Tukey's IQR Rule**:
- $IQR = Q_3 - Q_1$
- Lower Bound $= Q_1 - 1.5 \times IQR$
- Upper Bound $= Q_3 + 1.5 \times IQR$

In [5]:
# Sample dataset with extreme outliers
incomes = pd.Series([25000, 30000, 35000, 38000, 40000, 42000, 45000, 48000, 50000, 250000])

q1 = incomes.quantile(0.25)
q3 = incomes.quantile(0.75)
iqr = q3 - q1

lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

print(f"Q1: {q1:,.0f} | Q3: {q3:,.0f} | IQR: {iqr:,.0f}")
print(f"Valid Range: [{lower_bound:,.0f}, {upper_bound:,.0f}]")

# Identify outliers
is_outlier = (incomes < lower_bound) | (incomes > upper_bound)
print("Outliers identified:\n", incomes[is_outlier])

# Cleaned data
clean_incomes = incomes[~is_outlier]
print(f"Count before: {len(incomes)} | Count after filtering outliers: {len(clean_incomes)}")

Q1: 35,750 | Q3: 47,250 | IQR: 11,500
Valid Range: [18,500, 64,500]
Outliers identified:
 9    250000
dtype: int64
Count before: 10 | Count after filtering outliers: 9


---
### 4. Partitioning into Feature Matrix $X$ and Target Vector $y$

Before passing data to Scikit-Learn or PyTorch:
1. Separate features from the prediction target: $X = \text{df.drop}(columns=[target])$ and $y = \text{df}[target]$.
2. Extract the underlying NumPy arrays via `.values` or keep as DataFrame for pipeline integration.

In [6]:
# Drop non-feature temporary bin columns
df_final = df_encoded.drop(columns=['Income_Bracket', 'Income_Quartile'])

# Extract X and y
target_column = 'Purchased'
X = df_final.drop(columns=[target_column])
y = df_final[target_column]

print("Feature Matrix X (DataFrame):\n", X)
print("\nTarget Vector y (Series):\n", y)
print(f"\nFinal Shapes -> X: {X.shape}, y: {y.shape}")
print(f"Are all features numeric? {np.issubdtype(X.values.dtype, np.number)}")

Feature Matrix X (DataFrame):
    Income  City_London  City_Paris  Vehicle_Type_Sedan  Vehicle_Type_Truck  \
0   45000            0           1                   1                   0   
1   85000            1           0                   0                   0   
2   62000            0           0                   1                   0   
3   38000            0           1                   0                   1   
4  110000            0           0                   0                   0   

   Satisfaction_Rank  
0                  1  
1                  3  
2                  2  
3                  2  
4                  3  

Target Vector y (Series):
 0    0
1    1
2    1
3    0
4    1
Name: Purchased, dtype: int64

Final Shapes -> X: (5, 6), y: (5,)
Are all features numeric? True


### Module 02 Conclusion & Congratulations!
You have successfully completed **Module 02: Pandas for Machine Learning**!

Throughout these 6 notebooks, you mastered:
1. `pd.Series`, `pd.DataFrame`, and memory downcasting (`01_series_and_dataframe_fundamentals.ipynb`).
2. `.loc`, `.iloc`, boolean indexing, and eliminating `SettingWithCopyWarning` (`02_indexing_filtering_and_assignment.ipynb`).
3. Missing value analysis, statistical imputation, and string cleaning (`03_data_cleaning_and_missing_values.ipynb`).
4. `groupby()`, `.agg()`, `.transform()`, and pivot tables (`04_aggregations_grouping_and_pivot_tables.ipynb`).
5. Relational joins, concatenation, and time series rolling/lag features (`05_combining_datasets_and_timeseries.ipynb`).
6. One-hot encoding, ordinal mapping, binning, IQR outlier detection, and $X, y$ partitioning (`06_feature_engineering_with_pandas.ipynb`).

**Up Next:** Proceed to **Module 03: Matplotlib** to create publication-quality charts and diagnostic machine learning performance curves!